# MCCFR Class Unit Tests

**Proper unit tests with assertions and expected values**

These tests verify correctness of the MCCFR implementation.

In [ ]:
import sys
import os

# Add parent directory to path (deep_CFR_vNB_integration folder)
current_dir = os.getcwd()
if current_dir.endswith('tests'):
    parent_dir = os.path.dirname(current_dir)
else:
    parent_dir = current_dir

sys.path.insert(0, parent_dir)

import random
import numpy as np
from core.mccfr import MCCFR
from custom_engine import (
    FoldAction, CallAction, CheckAction, RaiseAction, DiscardAction, TerminalState
)

# Test tracking
tests_passed = 0
tests_failed = 0

def run_test(test_name, test_func):
    """Run a test and track results"""
    global tests_passed, tests_failed
    try:
        test_func()
        print(f"✓ {test_name} PASSED")
        tests_passed += 1
    except AssertionError as e:
        print(f"✗ {test_name} FAILED: {e}")
        tests_failed += 1
    except Exception as e:
        print(f"✗ {test_name} ERROR: {e}")
        tests_failed += 1

## 1. Initial State Tests

In [ ]:
def test_initial_state_properties():
    """Test that initial state has correct properties"""
    random.seed(42)  # For reproducibility
    mccfr = MCCFR()
    state = mccfr.create_initial_state()
    
    # Should start at street 0 (preflop)
    assert state.street == 0, f"Expected street 0, got {state.street}"
    
    # Both players should have 3 cards
    assert len(state.hands[0]) == 3, f"Player 0 should have 3 cards, got {len(state.hands[0])}"
    assert len(state.hands[1]) == 3, f"Player 1 should have 3 cards, got {len(state.hands[1])}"
    
    # Board should be empty at start
    assert len(state.board) == 0, f"Board should be empty at start, got {len(state.board)} cards"
    
    # Should not be terminal (check type)
    assert not isinstance(state, TerminalState), "Initial state should not be terminal"
    
    # Button should be 0 or 1
    assert state.button in [0, 1], f"Button should be 0 or 1, got {state.button}"

run_test("Initial state properties", test_initial_state_properties)

## 2. Infoset Format Tests

In [ ]:
def test_infoset_string_format():
    """Test that infoset string has correct format"""
    mccfr = MCCFR()
    state = mccfr.create_initial_state()
    infoset = mccfr.get_infoset(state, 0)
    
    # Should have format: "S{street}|H:{hand}|B:{board}|A:{history}"
    parts = infoset.split('|')
    assert len(parts) == 4, f"Infoset should have 4 parts separated by |, got {len(parts)}"
    
    # Check each part starts with correct prefix
    assert parts[0].startswith('S'), f"First part should start with 'S', got '{parts[0]}'"
    assert parts[1].startswith('H:'), f"Second part should start with 'H:', got '{parts[1]}'"
    assert parts[2].startswith('B:'), f"Third part should start with 'B:', got '{parts[2]}'"
    assert parts[3].startswith('A:'), f"Fourth part should start with 'A:', got '{parts[3]}'"
    
    # Street should be a number 0-3
    street_num = int(parts[0][1:])
    assert 0 <= street_num <= 3, f"Street should be 0-3, got {street_num}"

run_test("Infoset string format", test_infoset_string_format)

In [ ]:
def test_infoset_empty_board_at_preflop():
    """Test that infoset has empty board at preflop"""
    mccfr = MCCFR()
    state = mccfr.create_initial_state()
    infoset = mccfr.get_infoset(state, 0)
    
    parts = infoset.split('|')
    board_part = parts[2]  # "B:{board}"
    
    # At preflop, board should be "B:" with nothing after colon
    assert board_part == "B:", f"Expected empty board 'B:', got '{board_part}'"

run_test("Infoset empty board at preflop", test_infoset_empty_board_at_preflop)

## 3. Action Key Mapping Tests

In [ ]:
def test_action_to_key_basic_actions():
    """Test that basic actions map to correct keys"""
    mccfr = MCCFR()
    state = mccfr.create_initial_state()
    player = 0
    
    # Test basic action mappings
    fold = FoldAction()
    call = CallAction()
    check = CheckAction()
    
    assert mccfr.action_to_key(fold, state, player) == "FOLD", "Fold should map to 'FOLD'"
    assert mccfr.action_to_key(call, state, player) == "CALL", "Call should map to 'CALL'"
    assert mccfr.action_to_key(check, state, player) == "CHECK", "Check should map to 'CHECK'"

run_test("Action to key - basic actions", test_action_to_key_basic_actions)

In [ ]:
def test_raise_action_key_format():
    """Test that raise actions have correct key format"""
    mccfr = MCCFR()
    state = mccfr.create_initial_state()
    player = 0
    
    # Get a raise action
    legal_actions = mccfr.get_legal_actions_list(state)
    raise_actions = [a for a in legal_actions if isinstance(a, RaiseAction)]
    
    if raise_actions:
        raise_action = raise_actions[0]
        key = mccfr.action_to_key(raise_action, state, player)
        
        # Should start with one of the raise size categories
        assert key.startswith('RAISE_SMALL') or key.startswith('RAISE_MEDIUM') or key.startswith('RAISE_LARGE'), \
            f"Raise key should start with RAISE_SMALL/MEDIUM/LARGE, got '{key}'"

run_test("Raise action key format", test_raise_action_key_format)

## 4. Legal Actions Tests

In [ ]:
def test_no_discards_at_preflop():
    """Test that discard actions are not legal at preflop"""
    mccfr = MCCFR()
    state = mccfr.create_initial_state()
    
    legal_actions = mccfr.get_legal_actions_list(state)
    discard_actions = [a for a in legal_actions if isinstance(a, DiscardAction)]
    
    assert len(discard_actions) == 0, f"No discards should be legal at preflop, got {len(discard_actions)}"

run_test("No discards at preflop", test_no_discards_at_preflop)

In [ ]:
def test_betting_actions_at_preflop():
    """Test that betting actions are legal at preflop"""
    mccfr = MCCFR()
    state = mccfr.create_initial_state()
    
    legal_actions = mccfr.get_legal_actions_list(state)
    betting_actions = [a for a in legal_actions 
                      if isinstance(a, (CallAction, RaiseAction, FoldAction))]
    
    assert len(betting_actions) > 0, "Should have betting actions at preflop"
    
    # Should have at least fold and call
    has_fold = any(isinstance(a, FoldAction) for a in legal_actions)
    has_call = any(isinstance(a, CallAction) for a in legal_actions)
    
    assert has_fold or has_call, "Should have at least fold or call action"

run_test("Betting actions at preflop", test_betting_actions_at_preflop)

## 5. Regret Matching Tests

In [ ]:
def test_regret_matching_positive_regrets():
    """Test regret matching with all positive regrets"""
    mccfr = MCCFR()
    state = mccfr.create_initial_state()
    legal_actions = mccfr.get_legal_actions_list(state)
    
    # Use first 3 actions: setup regrets 3, 2, 1 -> should normalize to 3/6, 2/6, 1/6
    action_keys = [mccfr.action_to_key(a, state, 0) for a in legal_actions[:3]]
    regrets = {action_keys[0]: 3.0, action_keys[1]: 2.0, action_keys[2]: 1.0}
    
    strategy = mccfr.regret_matching(regrets, legal_actions[:3], state, 0)
    
    # Check probabilities
    assert abs(strategy[action_keys[0]] - 0.5) < 0.001, f"Strategy should be 0.5, got {strategy[action_keys[0]]}"
    assert abs(strategy[action_keys[1]] - 0.333) < 0.01, f"Strategy should be ~0.333, got {strategy[action_keys[1]]}"
    assert abs(strategy[action_keys[2]] - 0.167) < 0.01, f"Strategy should be ~0.167, got {strategy[action_keys[2]]}"
    
    # Should sum to 1
    total = sum(strategy.values())
    assert abs(total - 1.0) < 0.001, f"Strategy should sum to 1.0, got {total}"

run_test("Regret matching - positive regrets", test_regret_matching_positive_regrets)

In [ ]:
def test_regret_matching_negative_regrets():
    """Test that negative regrets are set to zero"""
    mccfr = MCCFR()
    state = mccfr.create_initial_state()
    legal_actions = mccfr.get_legal_actions_list(state)
    
    # Regrets: 3, -2, 1 -> negative set to 0, then normalize to 3/4, 0, 1/4
    action_keys = [mccfr.action_to_key(a, state, 0) for a in legal_actions[:3]]
    regrets = {action_keys[0]: 3.0, action_keys[1]: -2.0, action_keys[2]: 1.0}
    strategy = mccfr.regret_matching(regrets, legal_actions[:3], state, 0)
    
    assert abs(strategy[action_keys[0]] - 0.75) < 0.001, f"Strategy should be 0.75, got {strategy[action_keys[0]]}"
    assert abs(strategy[action_keys[1]] - 0.0) < 0.001, f"Strategy should be 0.0, got {strategy[action_keys[1]]}"
    assert abs(strategy[action_keys[2]] - 0.25) < 0.001, f"Strategy should be 0.25, got {strategy[action_keys[2]]}"
    
    # Should sum to 1
    total = sum(strategy.values())
    assert abs(total - 1.0) < 0.001, f"Strategy should sum to 1.0, got {total}"

run_test("Regret matching - negative regrets", test_regret_matching_negative_regrets)

In [ ]:
def test_regret_matching_all_negative():
    """Test uniform strategy when all regrets are negative"""
    mccfr = MCCFR()
    state = mccfr.create_initial_state()
    legal_actions = mccfr.get_legal_actions_list(state)
    
    # All negative -> should give uniform distribution
    action_keys = [mccfr.action_to_key(a, state, 0) for a in legal_actions[:3]]
    regrets = {action_keys[0]: -1.0, action_keys[1]: -2.0, action_keys[2]: -3.0}
    strategy = mccfr.regret_matching(regrets, legal_actions[:3], state, 0)
    
    expected = 1.0 / 3.0
    for action, prob in strategy.items():
        assert abs(prob - expected) < 0.001, \
            f"All negative regrets should give uniform distribution, got {prob} for {action}"
    
    # Should sum to 1
    total = sum(strategy.values())
    assert abs(total - 1.0) < 0.001, f"Strategy should sum to 1.0, got {total}"

run_test("Regret matching - all negative", test_regret_matching_all_negative)

## 6. External Sampling Tests

In [ ]:
def test_external_sampling_returns_valid_utility():
    """Test that external_sampling returns valid utility value"""
    random.seed(123)
    mccfr = MCCFR()
    
    # Run iterations - external_sampling returns utility for traversing player only
    for i in range(5):
        state = mccfr.create_initial_state()
        utility = mccfr.external_sampling(state, i % 2)
        
        # Should return a valid float (not NaN, not infinite)
        assert isinstance(utility, (int, float)), f"Utility should be numeric, got {type(utility)}"
        assert not np.isnan(utility), f"Utility should not be NaN"
        assert not np.isinf(utility), f"Utility should not be infinite"

run_test("External sampling returns valid utility", test_external_sampling_returns_valid_utility)

## 7. Regret Table Accumulation Tests

In [ ]:
def test_regret_table_grows_with_iterations():
    """Test that regret table accumulates entries over iterations"""
    random.seed(456)
    mccfr = MCCFR()
    
    # Start with empty table
    assert len(mccfr.regret_table) == 0, "Regret table should start empty"
    
    # Run iterations
    for i in range(5):
        state = mccfr.create_initial_state()
        mccfr.external_sampling(state, i % 2)
    
    # Should have accumulated some regrets
    assert len(mccfr.regret_table) > 0, "Regret table should have entries after iterations"

run_test("Regret table grows with iterations", test_regret_table_grows_with_iterations)

In [ ]:
def test_regret_values_are_valid():
    """Test that regret values are valid (not NaN, not infinite)"""
    random.seed(789)
    mccfr = MCCFR()
    
    # Run some iterations
    for i in range(10):
        state = mccfr.create_initial_state()
        mccfr.external_sampling(state, i % 2)
    
    # Check all regrets are valid
    for infoset, regrets in mccfr.regret_table.items():
        for action, regret in regrets.items():
            assert not np.isnan(regret), f"Regret is NaN for infoset '{infoset}', action '{action}'"
            assert not np.isinf(regret), f"Regret is infinite for infoset '{infoset}', action '{action}'"

run_test("Regret values are valid", test_regret_values_are_valid)

## 8. Action Selection Tests

In [ ]:
def test_select_action_returns_legal_action():
    """Test that select_action always returns a legal action"""
    random.seed(101)
    mccfr = MCCFR()
    
    # Train a bit first
    for i in range(5):
        state = mccfr.create_initial_state()
        mccfr.external_sampling(state, i % 2)
    
    # Now test action selection
    state = mccfr.create_initial_state()
    legal_actions = mccfr.get_legal_actions_list(state)
    # select_action returns an action instance, not a key
    legal_action_types = {type(a).__name__ for a in legal_actions}
    
    # Select action multiple times
    for _ in range(10):
        selected_action = mccfr.select_action(state, 0)
        assert selected_action is not None, "Selected action should not be None"
        selected_type = type(selected_action).__name__
        assert selected_type in legal_action_types, \
            f"Selected action type '{selected_type}' not in legal action types: {legal_action_types}"

run_test("Select action returns legal action", test_select_action_returns_legal_action)

## 9. Deterministic Tests

In [ ]:
def test_infoset_format_consistency():
    """Test that infoset format is consistent across runs"""
    
    # First run
    random.seed(999)
    mccfr1 = MCCFR()
    state1 = mccfr1.create_initial_state()
    infoset1 = mccfr1.get_infoset(state1, 0)
    
    # Second run with same seed
    random.seed(999)
    mccfr2 = MCCFR()
    state2 = mccfr2.create_initial_state()
    infoset2 = mccfr2.get_infoset(state2, 0)
    
    # Infosets should have same format
    parts1 = infoset1.split('|')
    parts2 = infoset2.split('|')
    
    assert len(parts1) == len(parts2) == 4, "Both infosets should have 4 parts"
    assert parts1[0] == parts2[0], "Street should be same (both preflop)"
    # Cards may differ due to deck shuffle, but format should be same
    assert parts1[1].startswith('H:') and parts2[1].startswith('H:'), "Both should have hand"
    assert parts1[2] == parts2[2] == 'B:', "Both should have empty board at preflop"
    assert parts1[3] == parts2[3] == 'A:', "Both should have empty action history"

run_test("Infoset format consistency", test_infoset_format_consistency)

## Test Summary

In [ ]:
print("\n" + "=" * 70)
print("MCCFR CLASS UNIT TEST SUMMARY")
print("=" * 70)
print(f"\nTests passed: {tests_passed}")
print(f"Tests failed: {tests_failed}")
print(f"Total tests: {tests_passed + tests_failed}")

if tests_failed == 0:
    print("\n✓ ALL TESTS PASSED!")
else:
    print(f"\n✗ {tests_failed} TEST(S) FAILED")
    
print("=" * 70)